# 与信審査② — 改善：モデルの探索・閾値・3段振り分け・説明・公平性

出典：第01章 05〜06節／パス：`01_credit/02_improved.ipynb`／パッケージ：numpy・pandas・scikit-learn・optuna・lightgbm・xgboost・matplotlib／実行時間の目安：約5分

このノート単体で実行できる。データの取得と切替は `01_baseline.ipynb` と同じ。
モデル・閾値・帯の境界は、すべて学習データの中の層化5分割CV（OOF予測）で選ぶ。テストは最後に1回だけ使い、結果を見て選び直さない。

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_val_predict, cross_val_score)

RS = 0


def make_credit_data(n: int, seed: int = 42) -> pd.DataFrame:
    """German Credit と同じ列（A1〜A20）の合成代替。値の分布は実データの近似、貸倒との関係は仮置き。"""
    r = np.random.default_rng(seed)

    def cat(codes, p):
        p = np.asarray(p, dtype=float)
        return r.choice(codes, size=n, p=p / p.sum())

    d = pd.DataFrame({
        "A1": cat(["A11", "A12", "A13", "A14"], [27, 27, 6, 40]),
        "A2": np.clip(np.round(r.gamma(3.0, 7.0, size=n)), 4, 72).astype(int),
        "A3": cat(["A30", "A31", "A32", "A33", "A34"], [4, 5, 53, 9, 29]),
        "A4": cat(["A40", "A41", "A42", "A43", "A44", "A45", "A46", "A48", "A49", "A410"],
                  [23, 10, 18, 28, 1, 2, 5, 1, 10, 1]),
        "A5": np.clip(np.round(np.exp(r.normal(7.8, 0.75, size=n))), 250, 18424).astype(int),
        "A6": cat(["A61", "A62", "A63", "A64", "A65"], [60, 10, 6, 5, 18]),
        "A7": cat(["A71", "A72", "A73", "A74", "A75"], [6, 17, 34, 17, 25]),
        "A8": cat([1, 2, 3, 4], [14, 23, 16, 47]),
        "A9": cat(["A91", "A92", "A93", "A94"], [5, 31, 55, 9]),
        "A10": cat(["A101", "A102", "A103"], [91, 4, 5]),
        "A11": cat([1, 2, 3, 4], [13, 31, 15, 41]),
        "A12": cat(["A121", "A122", "A123", "A124"], [28, 23, 33, 15]),
        "A13": np.clip(np.round(r.gamma(2.5, 6.6, size=n) + 19), 19, 75).astype(int),
        "A14": cat(["A141", "A142", "A143"], [14, 5, 81]),
        "A15": cat(["A151", "A152", "A153"], [18, 71, 11]),
        "A16": cat([1, 2, 3, 4], [63, 33, 3, 1]),
        "A17": cat(["A171", "A172", "A173", "A174"], [2, 20, 63, 15]),
        "A18": cat([1, 2], [85, 15]),
        "A19": cat(["A191", "A192"], [60, 40]),
        "A20": cat(["A201", "A202"], [96, 4]),
    })
    # 貸倒の起こりやすさ（仮置き）：口座状態・期間・返済履歴・金額・貯蓄・資産・他ローンで決まる
    logit = (
        -1.05
        + d["A1"].map({"A11": 1.0, "A12": 0.5, "A13": -0.2, "A14": -1.2})
        + 0.03 * (d["A2"] - 21)
        + d["A3"].map({"A30": 1.0, "A31": 0.9, "A32": 0.0, "A33": -0.2, "A34": -0.7})
        + 0.00008 * (d["A5"] - 3271)
        + d["A6"].map({"A61": 0.3, "A62": 0.0, "A63": -0.3, "A64": -0.5, "A65": -0.4})
        + 0.4 * (d["A12"] == "A124") + 0.4 * (d["A14"] == "A141")
        + 0.15 * (d["A8"] - 3)
    )
    d["default"] = (r.uniform(size=n) < 1.0 / (1.0 + np.exp(-logit))).astype(int)
    return d


GERMAN_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
GERMAN_COLS = ["A1", "A2", "A3", "A4", "A5", "A6", "A7", "A8", "A9", "A10",
               "A11", "A12", "A13", "A14", "A15", "A16", "A17", "A18", "A19", "A20", "klass"]
GERMAN_NUM = ["A2", "A5", "A8", "A11", "A13", "A16", "A18"]
# 機微な属性（事前指定で特徴量から外す）：A9=性別・婚姻状況、A13=年齢、A20=外国人労働者
SENSITIVE = ["A9", "A13", "A20"]


def load_german() -> pd.DataFrame:
    g = pd.read_csv(GERMAN_URL, sep=r"\s+", header=None, names=GERMAN_COLS)
    g["default"] = (g["klass"] == 2).astype(int)  # 2=信用リスクが悪い（bad）を1とする
    return g.drop(columns=["klass"])


INPUT = "input/applications.csv"
try:
    df = load_german()
    print("公開データを使用: German Credit（個人与信の実データ）")
except Exception:
    try:
        df = pd.read_csv(INPUT)
        print(f"Inputを使用: {INPUT}（German Credit形式の合成代替）")
    except FileNotFoundError:
        df = make_credit_data(1000)
        print("Inputが無いため合成データを生成（German Credit形式）")
FEATS = [c for c in df.columns if c not in SENSITIVE + ["default"]]
num_cols = [c for c in FEATS if c in GERMAN_NUM]
cat_cols = [c for c in FEATS if c not in GERMAN_NUM]
print(f"件数: {len(df)}, 貸倒率: {df['default'].mean():.3f} / 特徴量 {len(FEATS)}列（機微な属性{len(SENSITIVE)}列を除外）")

X = df.drop(columns=["default"])
y = df["default"]
# 最終評価用のテストを先に分ける（調整には使わない）。検証は学習データの中の層化5分割CVで行う。
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RS
)
pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
print(f"学習: {len(X_tr)}件（検証は学習内の5分割CV） / テスト: {len(X_te)}件")
print(f"貸倒率 学習:{y_tr.mean():.3f} テスト:{y_te.mean():.3f}")

# ベースライン（線形の物差し）。改善の比較はすべて学習データの中の層化5分割CVで行う
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)
lr = make_pipeline(clone(pre), LogisticRegression(max_iter=5000))
s = cross_val_score(lr, X_tr, y_tr, cv=cv, scoring="average_precision")
print(f"[ベースライン・線形] CVのPR-AUC={s.mean():.4f}（±{s.std(ddof=1):.4f}）")

公開データを使用: German Credit（個人与信の実データ）
件数: 1000, 貸倒率: 0.300 / 特徴量 17列（機微な属性3列を除外）
学習: 750件（検証は学習内の5分割CV） / テスト: 250件
貸倒率 学習:0.300 テスト:0.300
[ベースライン・線形] CVのPR-AUC=0.5767（±0.0845）


In [2]:
# 自動探索：モデル選択そのものを探索対象にする（学習データの中の5分割CVで選ぶ。テストは使わない）
# パラメータ名はモデルごとに接頭辞を付ける（同名で選択肢が違うとOptunaが壊れるため）
import warnings
warnings.filterwarnings("ignore", message=".*IProgress.*")
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


def build_model(params: dict):
    kind = params["model"]
    if kind == "logreg":
        est = LogisticRegression(C=params["logreg_C"], max_iter=5000)
    elif kind == "rf":
        est = RandomForestClassifier(n_estimators=200, max_depth=params["rf_max_depth"],
                                     min_samples_leaf=params["rf_min_samples_leaf"],
                                     max_features=params["rf_max_features"], random_state=RS, n_jobs=1)
    elif kind == "hgb":
        est = HistGradientBoostingClassifier(
            max_iter=params["hgb_max_iter"], learning_rate=params["hgb_learning_rate"],
            max_leaf_nodes=params["hgb_max_leaf_nodes"], min_samples_leaf=params["hgb_min_samples_leaf"],
            random_state=RS)
    elif kind == "lgbm":
        import lightgbm as lgb
        est = lgb.LGBMClassifier(
            n_estimators=params["lgbm_n_estimators"], learning_rate=params["lgbm_learning_rate"],
            num_leaves=params["lgbm_num_leaves"], min_child_samples=params["lgbm_min_child_samples"],
            random_state=RS, verbosity=-1)
    else:
        import xgboost as xgb
        est = xgb.XGBClassifier(
            n_estimators=params["xgb_n_estimators"], learning_rate=params["xgb_learning_rate"],
            max_depth=params["xgb_max_depth"], min_child_weight=params["xgb_min_child_weight"],
            random_state=RS, n_jobs=1, eval_metric="logloss")
    return make_pipeline(clone(pre), est)


def objective(trial):
    # 連続値の範囲で探す（選択肢だけの狭い空間だと、同じ構成を何度も試して予算を無駄にする）
    kind = trial.suggest_categorical("model", ["logreg", "rf", "hgb", "lgbm", "xgb"])
    params = {"model": kind}
    if kind == "logreg":
        params["logreg_C"] = trial.suggest_float("logreg_C", 1e-3, 10.0, log=True)
    elif kind == "rf":
        params["rf_max_depth"] = trial.suggest_int("rf_max_depth", 3, 16)
        params["rf_min_samples_leaf"] = trial.suggest_int("rf_min_samples_leaf", 1, 30)
        params["rf_max_features"] = trial.suggest_float("rf_max_features", 0.1, 0.6)
    elif kind == "hgb":
        params["hgb_max_iter"] = trial.suggest_int("hgb_max_iter", 50, 300)
        params["hgb_learning_rate"] = trial.suggest_float("hgb_learning_rate", 0.01, 0.2, log=True)
        params["hgb_max_leaf_nodes"] = trial.suggest_int("hgb_max_leaf_nodes", 4, 31)
        params["hgb_min_samples_leaf"] = trial.suggest_int("hgb_min_samples_leaf", 10, 60)
    elif kind == "lgbm":
        params["lgbm_n_estimators"] = trial.suggest_int("lgbm_n_estimators", 50, 300)
        params["lgbm_learning_rate"] = trial.suggest_float("lgbm_learning_rate", 0.01, 0.2, log=True)
        params["lgbm_num_leaves"] = trial.suggest_int("lgbm_num_leaves", 4, 31)
        params["lgbm_min_child_samples"] = trial.suggest_int("lgbm_min_child_samples", 10, 60)
    else:
        params["xgb_n_estimators"] = trial.suggest_int("xgb_n_estimators", 50, 300)
        params["xgb_learning_rate"] = trial.suggest_float("xgb_learning_rate", 0.01, 0.2, log=True)
        params["xgb_max_depth"] = trial.suggest_int("xgb_max_depth", 2, 6)
        params["xgb_min_child_weight"] = trial.suggest_float("xgb_min_child_weight", 1.0, 10.0, log=True)
    return cross_val_score(build_model(params), X_tr, y_tr, cv=cv, scoring="average_precision").mean()


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=30, show_progress_bar=False)
n_unique = len({tuple(sorted(t.params.items())) for t in study.trials})
print(f"試行数: {len(study.trials)}（異なる構成 {n_unique}） / 最良（CV）: {study.best_value:.4f}"
      f" / 最良のモデル: {study.best_params['model']}")
rows = sorted(((t.number, t.params["model"], t.value) for t in study.trials), key=lambda r: -r[2])[:5]
print("上位5試行（番号/モデル/CVのPR-AUC）:")
for n, md, v in rows:
    print(f"  {n}: {md} {v:.4f}")

# 探索で選んだ値は楽観的に偏る。乱数を変えた繰返しCV（5分割×3回）で、線形と同じ条件で測り直す
cv2 = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=1)
cand = build_model(study.best_params)
s_lr = cross_val_score(lr, X_tr, y_tr, cv=cv2, scoring="average_precision")
s_cd = cross_val_score(cand, X_tr, y_tr, cv=cv2, scoring="average_precision")
se = (s_cd - s_lr).std(ddof=1) / np.sqrt(len(s_lr))
print(f"[測り直し・繰返しCV] 線形 {s_lr.mean():.4f} / 探索最良 {s_cd.mean():.4f}"
      f"（差 {s_cd.mean() - s_lr.mean():+.4f}、標準誤差 {se:.4f}）")

# 確率を金額判断に使うなら、順位（PR-AUC）だけでなく確率の目盛り（較正）を確かめる。CVの予測（OOF）で測る
cand_cal = CalibratedClassifierCV(estimator=cand, cv=5)
oof_lr = cross_val_predict(lr, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
oof_raw = cross_val_predict(cand, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
oof_cd = cross_val_predict(cand_cal, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
print(f"[OOF] 線形 Brier={brier_score_loss(y_tr, oof_lr):.4f}"
      f" / 探索最良 較正前 Brier={brier_score_loss(y_tr, oof_raw):.4f}"
      f" → 較正後 Brier={brier_score_loss(y_tr, oof_cd):.4f}（PR-AUC {average_precision_score(y_tr, oof_cd):.4f}）")
assert np.isfinite(oof_cd).all()

試行数: 30（異なる構成 30） / 最良（CV）: 0.6390 / 最良のモデル: hgb
上位5試行（番号/モデル/CVのPR-AUC）:
  25: hgb 0.6390
  2: lgbm 0.6332
  24: hgb 0.6318
  27: hgb 0.6300
  8: lgbm 0.6289


[測り直し・繰返しCV] 線形 0.5895 / 探索最良 0.6327（差 +0.0431、標準誤差 0.0161）


[OOF] 線形 Brier=0.1742 / 探索最良 較正前 Brier=0.2172 → 較正後 Brier=0.1658（PR-AUC 0.6203）


In [3]:
# 閾値を費用で決める。見逃し（FN: 貸倒）10万円、空振り（FP: 機会損失）2万円と仮置きする（5:1）
COST_FN, COST_FP = 10.0, 2.0  # 単位：万円
TS = np.round(np.arange(0.02, 0.81, 0.02), 2)  # 候補は細かく取り、最小が端に来ないことを確かめる
yv = y_tr.to_numpy()


def exp_loss(y, p, t):
    deny = p >= t
    return (((~deny) & (y == 1)).sum() * COST_FN + (deny & (y == 0)).sum() * COST_FP) / len(y)


# 物差し：モデルを使わない2つの方針
print(f"[OOF] 全件承認 {exp_loss(yv, np.zeros(len(yv)), 0.5):.3f} / 全件否決 {exp_loss(yv, np.ones(len(yv)), 0.5):.3f}（万円/件）")
best = {}
print("閾値 / 期待損失（線形） / 期待損失（探索最良・較正後）［OOF］:")
for name, p in [("logreg", oof_lr), ("cand", oof_cd)]:
    losses = [exp_loss(yv, p, t) for t in TS]
    best[name] = (float(TS[int(np.argmin(losses))]), min(losses))
for t in [0.06, 0.10, 0.14, 0.18, 0.22, 0.30, 0.40, 0.50]:
    print(f"{t:.2f} / {exp_loss(yv, oof_lr, t):.3f} / {exp_loss(yv, oof_cd, t):.3f}")
print(f"最小損失: 線形 閾値{best['logreg'][0]:.2f}（{best['logreg'][1]:.3f}）"
      f" / 探索最良 閾値{best['cand'][0]:.2f}（{best['cand'][1]:.3f}）［OOFで選択］")
assert TS[0] < best["logreg"][0] < TS[-1] and TS[0] < best["cand"][0] < TS[-1]
# 採用の規則（事前に決める）：業務の物差し（OOFの期待損失）で線形に勝つときだけ、複雑なモデルを採用する
ADOPT = "cand" if best["cand"][1] < best["logreg"][1] else "logreg"
T_STAR = best[ADOPT][0]
print(f"採用: {'探索最良' if ADOPT == 'cand' else '線形（ベースラインのモデル）'}＋閾値{T_STAR:.2f}")

[OOF] 全件承認 3.000 / 全件否決 1.400（万円/件）
閾値 / 期待損失（線形） / 期待損失（探索最良・較正後）［OOF］:
0.06 / 1.187 / 1.371
0.10 / 1.125 / 1.269
0.14 / 1.072 / 1.168
0.18 / 1.091 / 1.128
0.22 / 1.187 / 1.131
0.30 / 1.235 / 1.243
0.40 / 1.504 / 1.571
0.50 / 1.899 / 1.941
最小損失: 線形 閾値0.16（1.061） / 探索最良 閾値0.20（1.120）［OOFで選択］
採用: 線形（ベースラインのモデル）＋閾値0.16


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
# threshold sweep on out-of-fold predictions (training data only)
plt.figure()
plt.plot(TS, [exp_loss(yv, oof_lr, t) for t in TS], marker=".", label="logreg")
plt.plot(TS, [exp_loss(yv, oof_cd, t) for t in TS], marker=".", label="search best (calibrated)")
plt.axhline(exp_loss(yv, np.zeros(len(yv)), 0.5), color="gray", ls="--", label="approve all")
plt.axhline(exp_loss(yv, np.ones(len(yv)), 0.5), color="gray", ls=":", label="reject all")
plt.xlabel("threshold"); plt.ylabel("expected loss (10k JPY / applicant)"); plt.legend()
plt.title("threshold sweep (out-of-fold, training data)")
plt.savefig(f"{FIG}/code-result-j01-02.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j01-02.png")

saved code-result-j01-02.png


In [5]:
# 3段振り分け：確信度で自動承認／人手審査／自動否決に分ける。境界はOOFで、費用と審査キャパシティから決める
C_REVIEW, CAPACITY = 0.5, 0.30  # 仮置き：人手審査1件0.5万円、人手に回せるのは申込の30%まで
oof_ad = oof_cd if ADOPT == "cand" else oof_lr


def band_of(p, ta, td):
    return np.where(p >= td, "deny", np.where(p < ta, "approve", "manual"))


def band_loss(y, p, ta, td):
    # 人手審査は正しく判定できると仮置きし、審査費用だけを数える（楽観的な前提）
    b = band_of(p, ta, td)
    fn = ((b == "approve") & (y == 1)).sum()
    fp = ((b == "deny") & (y == 0)).sum()
    return (fn * COST_FN + fp * COST_FP + (b == "manual").sum() * C_REVIEW) / len(y)


cands = [(ta, td) for ta in TS[TS <= T_STAR] for td in TS[TS > T_STAR]
         if (band_of(oof_ad, ta, td) == "manual").mean() <= CAPACITY]
T_APPROVE, T_DENY = min(cands, key=lambda c: band_loss(yv, oof_ad, *c))
print(f"境界［OOFで選択］: 自動承認 < {T_APPROVE:.2f} ≦ 人手 < {T_DENY:.2f} ≦ 自動否決"
      f" / 期待損失 {band_loss(yv, oof_ad, T_APPROVE, T_DENY):.3f}万円/件（審査費用込み）")
band = band_of(oof_ad, T_APPROVE, T_DENY)
for b in ["approve", "manual", "deny"]:
    m = band == b
    print(f"{b:>7}: {m.mean():.3f} / 件数 {m.sum():4d} / 帯の貸倒率 {yv[m].mean():.3f}［OOF］")

境界［OOFで選択］: 自動承認 < 0.14 ≦ 人手 < 0.38 ≦ 自動否決 / 期待損失 0.790万円/件（審査費用込み）
approve: 0.364 / 件数  273 / 帯の貸倒率 0.092［OOF］
 manual: 0.300 / 件数  225 / 帯の貸倒率 0.280［OOF］
   deny: 0.336 / 件数  252 / 帯の貸倒率 0.544［OOF］


In [6]:
# 最終確認：ここで初めてテストを使う（1回だけ。結果を見て選び直さない）
lr.fit(X_tr, y_tr)
cand_cal.fit(X_tr, y_tr)
p_lr = lr.predict_proba(X_te)[:, 1]
p_cd = cand_cal.predict_proba(X_te)[:, 1]
p_ad = p_cd if ADOPT == "cand" else p_lr
yt = y_te.to_numpy()
ap_lr, ap_cd = average_precision_score(yt, p_lr), average_precision_score(yt, p_cd)
# テストは250件（貸倒75件）と小さい。差のぶれをブートストラップで示す
r = np.random.default_rng(RS)
diffs = []
for _ in range(1000):
    i = r.integers(0, len(yt), len(yt))
    diffs.append(average_precision_score(yt[i], p_cd[i]) - average_precision_score(yt[i], p_lr[i]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"[テスト] PR-AUC 線形 {ap_lr:.4f} / 探索最良 {ap_cd:.4f}（差 {ap_cd - ap_lr:+.4f}、95%区間 {lo:+.3f}〜{hi:+.3f}）")
policies = [
    ("全件承認", exp_loss(yt, np.zeros(len(yt)), 0.5)),
    ("全件否決", exp_loss(yt, np.ones(len(yt)), 0.5)),
    ("線形・閾値0.5（ベースライン）", exp_loss(yt, p_lr, 0.5)),
    (f"線形・閾値{best['logreg'][0]:.2f}", exp_loss(yt, p_lr, best["logreg"][0])),
    (f"探索最良・閾値{best['cand'][0]:.2f}", exp_loss(yt, p_cd, best["cand"][0])),
]
print("期待損失（万円/件）［テスト］:")
for name, v in policies:
    print(f"  {name}: {v:.3f}")
band_te = band_of(p_ad, T_APPROVE, T_DENY)
for b in ["approve", "manual", "deny"]:
    m = band_te == b
    print(f"{b:>7}: {m.mean():.3f} / 件数 {m.sum():4d} / 帯の貸倒率 {yt[m].mean():.3f}［テスト］")
print(f"[改善結果] 期待損失 {policies[2][1]:.3f} → {exp_loss(yt, p_ad, T_STAR):.3f}万円/件"
      f"（採用案＝{'探索最良' if ADOPT == 'cand' else '線形'}・閾値0.5→{T_STAR:.2f}、テスト）"
      f" / 3段（審査費用込み） {band_loss(yt, p_ad, T_APPROVE, T_DENY):.3f}")

[テスト] PR-AUC 線形 0.5999 / 探索最良 0.5307（差 -0.0692、95%区間 -0.163〜+0.022）
期待損失（万円/件）［テスト］:
  全件承認: 3.000
  全件否決: 1.400
  線形・閾値0.5（ベースライン）: 1.968
  線形・閾値0.16: 1.016
  探索最良・閾値0.20: 1.064
approve: 0.384 / 件数   96 / 帯の貸倒率 0.083［テスト］
 manual: 0.304 / 件数   76 / 帯の貸倒率 0.316［テスト］
   deny: 0.312 / 件数   78 / 帯の貸倒率 0.551［テスト］
[改善結果] 期待損失 1.968 → 1.016万円/件（採用案＝線形・閾値0.5→0.16、テスト） / 3段（審査費用込み） 0.752


In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
from sklearn.metrics import precision_recall_curve
# band default rates (test)
rates = [yt[band_te == b].mean() for b in ["approve", "manual", "deny"]]
plt.figure(); plt.bar(["approve", "manual", "deny"], rates)
plt.ylabel("default rate"); plt.title("band default rates (test)")
plt.savefig(f"{FIG}/code-result-j01-03.png", dpi=150, bbox_inches="tight"); plt.close()
# PR curves: baseline vs search best (test)
plt.figure()
for name, p in [("logreg (baseline)", p_lr), ("search best (calibrated)", p_cd)]:
    prec, rec, _ = precision_recall_curve(yt, p)
    plt.step(rec, prec, where="post", label=name)
plt.xlabel("recall"); plt.ylabel("precision"); plt.legend(); plt.title("PR curves (test)")
plt.savefig(f"{FIG}/code-result-j01-04.png", dpi=150, bbox_inches="tight"); plt.close()
# expected loss by policy (test)
labels = ["approve\nall", "reject\nall", "logreg\n@0.5", f"logreg\n@{best['logreg'][0]:.2f}",
          f"search best\n@{best['cand'][0]:.2f}"]
plt.figure(figsize=(7, 4)); plt.bar(labels, [v for _, v in policies])
plt.ylabel("expected loss (10k JPY / applicant)"); plt.title("expected loss by policy (test)")
plt.savefig(f"{FIG}/code-result-j01-05.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j01-03.png, code-result-j01-04.png, code-result-j01-05.png")

saved code-result-j01-03.png, code-result-j01-04.png, code-result-j01-05.png


In [8]:
# 全体の説明：どの特徴が効いたか（順列重要度。CVの各分割で、学習に使っていない側で測って平均する）
model_ad = cand if ADOPT == "cand" else lr
imps = []
for k_tr, k_va in cv.split(X_tr, y_tr):
    m = clone(model_ad).fit(X_tr.iloc[k_tr], y_tr.iloc[k_tr])
    res = permutation_importance(m, X_tr.iloc[k_va][FEATS], y_tr.iloc[k_va], n_repeats=5,
                                 random_state=RS, scoring="average_precision")
    imps.append(res.importances_mean)
imp = pd.Series(np.mean(imps, axis=0), index=FEATS).sort_values(ascending=False)
print("順列重要度 上位6［CV］:")
for c, v in imp.head(6).items():
    print(f"{c:>4}: {v:.4f}")

# 個別の説明：否決1件の理由素。列を1つずつ「学習データの代表値」に置き換え、確率がどれだけ下がるかで測る
# 機微な属性はモデルに入れていないので、理由にも出てこない
i = int(np.where((band == "deny") & (yv == 1))[0][0])
row = X_tr.iloc[[i]]
p0 = oof_ad[i]
m_all = clone(model_ad).fit(X_tr, y_tr)
p_in = m_all.predict_proba(row)[0, 1]
drops = {}
for c in FEATS:
    alt = row.copy()
    alt[c] = X_tr[c].median() if c in num_cols else X_tr[c].mode().iat[0]
    drops[c] = p_in - m_all.predict_proba(alt)[0, 1]
print(f"\n申込 {X_tr.index[i]}: 確率 {p0:.2f}（OOF） → 自動否決。理由素（確率の下がり幅）:")
for c, d in sorted(drops.items(), key=lambda kv: -kv[1])[:3]:
    print(f"- {c}: 申請値={row[c].iat[0]} → 代表値なら確率 -{d:.2f}")

# 特徴量選定の実演：重要度上位6列に絞って、同じモデルでCVを比べる（テストは使わない）
top6 = list(imp.head(6).index)
pre6 = ColumnTransformer([
    ("num", StandardScaler(), [c for c in top6 if c in num_cols]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), [c for c in top6 if c in cat_cols]),
])
m6 = make_pipeline(pre6, clone(model_ad.steps[-1][1]))
s_all = cross_val_score(model_ad, X_tr, y_tr, cv=cv, scoring="average_precision")
s_6 = cross_val_score(m6, X_tr[top6], y_tr, cv=cv, scoring="average_precision")
print(f"\n選択列: {sorted(top6)}")
print(f"[全{len(FEATS)}列] CVのPR-AUC={s_all.mean():.4f} / [選択6列] CVのPR-AUC={s_6.mean():.4f}")

順列重要度 上位6［CV］:
  A1: 0.0633
  A6: 0.0410
  A3: 0.0394
  A4: 0.0348
  A2: 0.0180
 A15: 0.0174

申込 925: 確率 0.90（OOF） → 自動否決。理由素（確率の下がり幅）:
- A1: 申請値=A11 → 代表値なら確率 -0.25
- A4: 申請値=A40 → 代表値なら確率 -0.16
- A3: 申請値=A31 → 代表値なら確率 -0.10

選択列: ['A1', 'A15', 'A2', 'A3', 'A4', 'A6']
[全17列] CVのPR-AUC=0.5767 / [選択6列] CVのPR-AUC=0.5889


In [9]:
# 公平性の確認①：機微な属性を入れたモデルと比べる（精度がほとんど変わらないなら、入れる理由はない）
all_cols = FEATS + SENSITIVE
pre_s = ColumnTransformer([
    ("num", StandardScaler(), [c for c in all_cols if c in GERMAN_NUM]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), [c for c in all_cols if c not in GERMAN_NUM]),
])
m_s = make_pipeline(pre_s, clone(model_ad.steps[-1][1]))
s_s = cross_val_score(m_s, X_tr[all_cols], y_tr, cv=cv, scoring="average_precision")
print(f"[CV] 機微な属性なし {s_all.mean():.4f} / あり {s_s.mean():.4f}（差 {s_s.mean() - s_all.mean():+.4f}）")

# 公平性の確認②：属性別の承認率と貸倒率（閾値で「貸す」と判断した割合。OOFの判断で測る）
# 属性を外しても、相関する列（期間・住居など）を通じて差が残ることがある。だから外した後も測る
approve = oof_ad < T_STAR
groups = {
    "女性（A92）": X_tr["A9"].to_numpy() == "A92",
    "男性（A91・A93・A94）": X_tr["A9"].to_numpy() != "A92",
    "25歳未満": X_tr["A13"].to_numpy() < 25,
    "25歳以上": X_tr["A13"].to_numpy() >= 25,
    "外国人労働者（A201）": X_tr["A20"].to_numpy() == "A201",
    "それ以外（A202）": X_tr["A20"].to_numpy() == "A202",
}
# 「返済した人のうち承認された割合」も並べる（返せる人を同じ割合で通しているか）
print("属性 / 件数 / 承認率 / 返済した人の承認率 / 実際の貸倒率［OOF］:")
for g, m in groups.items():
    print(f"{g}: {m.sum():4d} / {approve[m].mean():.3f} / {approve[m & (yv == 0)].mean():.3f} / {yv[m].mean():.3f}")

[CV] 機微な属性なし 0.5767 / あり 0.5824（差 +0.0057）
属性 / 件数 / 承認率 / 返済した人の承認率 / 実際の貸倒率［OOF］:
女性（A92）:  227 / 0.383 / 0.510 / 0.344
男性（A91・A93・A94）:  523 / 0.398 / 0.508 / 0.281
25歳未満:  105 / 0.229 / 0.349 / 0.400
25歳以上:  645 / 0.420 / 0.530 / 0.284
外国人労働者（A201）:  724 / 0.395 / 0.514 / 0.307
それ以外（A202）:   26 / 0.346 / 0.391 / 0.115


## 読み取りどころ

- 探索の最良（CV）は楽観的に偏る。乱数を変えた繰返しCVで、線形と同じ条件で測り直すこと。
- PR-AUCの勝ち負けと、期待損失（業務の物差し）の勝ち負けは別であること。採用は期待損失で決める。
- 閾値の最小が候補の端に来ていないこと。全件承認・全件否決より安いこと。
- 帯の境界は費用と審査キャパシティで決めること。
- 重要度・特徴量選定・公平性の確認も、テストではなくCVで行うこと。機微な属性は外しても、承認率の差は測り続けること。